In [ ]:
import os
import gc
import glob
import xarray as xr
import numpy as np
import pandas as pd
from multiprocessing import Pool
from eofs.xarray import Eof
import matplotlib.pyplot as plt
import cartopy.crs as ccrs
import cartopy.feature as cfeature
import matplotlib.colors as mcolors

# ================= 1. 核心配置区 =================
FIG_OUT_DIR = "/home/weiji/restart_exam/code/20260315NAM/resul/"
os.makedirs(FIG_OUT_DIR, exist_ok=True)

LAT_MIN_AO = 20.0
# NAM 现在也基于 20N-90N 的 EOF 进行计算，所以极帽平均的纬度配置被移除

# ---------------- 新增：目标 pressure levels ----------------
TARGET_PLEV_HPA = np.array([
    1000.0, 950.0, 900.0, 850.0, 800.0, 750.0,
    700.0, 600.0, 550.0, 500.0, 450.0, 400.0,
    350.0, 300.0, 250.0, 225.0, 200.0, 175.0,
    150.0, 125.0, 100.0, 70.0, 50.0, 30.0,
    20.0, 10.0, 7.0, 5.0, 3.0, 2.0, 1.0
], dtype=np.float64)

TARGET_PLEV_PA = TARGET_PLEV_HPA * 100.0
AO_PLEV_HPA = 1000.0

CORES = 10
SHIFT_RUN2_INTERNAL_YEAR_OFFSET = 104

# ================= 文件命名模板 =================
EXPERIMENTS = [
    {
        "name": "B2000WCN001002",
        "base_dir": "/mnt/soclim0/public_data/weiji/B2000WCN001002",
        "file_template": "B2000WCN.sample.cam.h3.{year:04d}.{var}.nc",
        "apply_shift": True,
        "is_reference": True,
        "use_reference": False
    },
    {
        "name": "B2000WCN_NOCOUPL001002",
        "base_dir": "/mnt/soclim0/public_data/weiji/B2000WCN_NOCOUPL001002",
        "file_template": "B2000WCN.NOCOUPL.sample.cam.h3.{year:04d}.{var}.nc",
        "apply_shift": True,
        "is_reference": False,
        "use_reference": False
    },
    {
        "name": "BWCN",
        "base_dir": "/mnt/soclim0/public_data/weiji/BWCN",
        "file_template": "BWCN.cam.h3.{year:04d}.{var}.nc",
        "apply_shift": False,
        "is_reference": False,
        "use_reference": True
    }
]

REF_DATA = {}

# ================= 2. 基础工具函数 =================
def shift_time_cftime(ds, year_offset):
    times = ds["time"].values
    new_times = [t.replace(year=t.year + year_offset) for t in times]
    ds["time"] = xr.DataArray(new_times, coords=[new_times], dims=["time"])
    return ds

def _process_wrapper(args):
    return process_one_year(*args)

def logp_interp_extrap_1d(p_prof, z_prof, target_p):
    p = np.asarray(p_prof, dtype=np.float64)
    z = np.asarray(z_prof, dtype=np.float64)
    tp = np.asarray(target_p, dtype=np.float64)

    mask = np.isfinite(p) & np.isfinite(z) & (p > 0)
    p = p[mask]
    z = z[mask]

    if p.size < 2:
        return np.full(tp.shape, np.nan, dtype=np.float64)

    order = np.argsort(p)
    p = p[order]
    z = z[order]

    lp = np.log(p)
    ltp = np.log(tp)

    out = np.interp(ltp, lp, z)

    mask_top = ltp < lp[0]
    if np.any(mask_top):
        slope_top = (z[1] - z[0]) / (lp[1] - lp[0])
        out[mask_top] = z[0] + slope_top * (ltp[mask_top] - lp[0])

    mask_bottom = ltp > lp[-1]
    if np.any(mask_bottom):
        slope_bottom = (z[-1] - z[-2]) / (lp[-1] - lp[-2])
        out[mask_bottom] = z[-1] + slope_bottom * (ltp[mask_bottom] - lp[-1])

    return out

def interp_to_pressure_levels(z3_da, p3d_da, target_p_pa, target_p_hpa):
    z_on_plev = xr.apply_ufunc(
        logp_interp_extrap_1d,
        p3d_da,
        z3_da,
        input_core_dims=[["lev"], ["lev"]],
        output_core_dims=[["plev"]],
        vectorize=True,
        dask="parallelized",
        kwargs={"target_p": target_p_pa},
        output_dtypes=[np.float64],
        dask_gufunc_kwargs={"output_sizes": {"plev": len(target_p_pa)}},
    )

    z_on_plev = z_on_plev.assign_coords(plev=("plev", target_p_hpa))
    z_on_plev["plev"].attrs["units"] = "hPa"
    z_on_plev["plev"].attrs["long_name"] = "pressure"

    z_on_plev = z_on_plev.transpose("time", "plev", "lat", "lon")
    z_on_plev = z_on_plev.rename({"plev": "lev"})
    z_on_plev["lev"].attrs["units"] = "hPa"
    z_on_plev["lev"].attrs["long_name"] = "pressure"

    return z_on_plev

def process_one_year(year_int, base_dir, apply_shift, file_template):
    f_z3_name = file_template.format(year=year_int, var="Z3")
    f_ps_name = file_template.format(year=year_int, var="PS")

    f_z3 = os.path.join(base_dir, "Z3", f_z3_name)
    if not os.path.exists(f_z3):
        f_z3 = os.path.join(base_dir, f_z3_name)

    f_ps = os.path.join(base_dir, "PS", f_ps_name)
    if not os.path.exists(f_ps):
        f_ps = os.path.join(base_dir, f_ps_name)

    if not (os.path.exists(f_z3) and os.path.exists(f_ps)):
        return None

    try:
        time_coder = xr.coders.CFDatetimeCoder(use_cftime=True)
        with xr.open_dataset(f_z3, decode_times=time_coder) as ds_z, \
             xr.open_dataset(f_ps, decode_times=time_coder) as ds_ps:

            p0 = 100000.0
            hyam = ds_z["hyam"]
            hybm = ds_z["hybm"]
            ps = ds_ps["PS"]

            p3d = hyam * p0 + hybm * ps

            # 只保留北半球 20N-90N，减少计算量
            z3_nh = ds_z["Z3"].sel(lat=slice(LAT_MIN_AO, 90.0))
            p3d_nh = p3d.sel(lat=slice(LAT_MIN_AO, 90.0))

            z3_plev_nh = interp_to_pressure_levels(
                z3_da=z3_nh,
                p3d_da=p3d_nh,
                target_p_pa=TARGET_PLEV_PA,
                target_p_hpa=TARGET_PLEV_HPA
            )

            # --- A: AO 用 1000 hPa ---
            da_z3_1000 = z3_plev_nh.sel(lev=AO_PLEV_HPA)
            da_z3_1000.name = "Z3_1000"

            # --- B: NAM 用 20N-90N 的纬向平均 (Zonal Mean) ---
            z3_zm = z3_plev_nh.mean("lon")
            z3_zm.name = "Z3_ZM"

            # --- 时间修正 ---
            if apply_shift and year_int >= 105:
                da_z3_1000 = shift_time_cftime(
                    da_z3_1000.to_dataset(name="Z3_1000"),
                    SHIFT_RUN2_INTERNAL_YEAR_OFFSET
                )["Z3_1000"]

                z3_zm = shift_time_cftime(
                    z3_zm.to_dataset(name="Z3_ZM"),
                    SHIFT_RUN2_INTERNAL_YEAR_OFFSET
                )["Z3_ZM"]

            out_1000 = da_z3_1000.compute()
            out_1000.name = "Z3_1000"

            out_zm = z3_zm.compute()
            out_zm.name = "Z3_ZM"
            out_zm["lev"].attrs["units"] = "hPa"
            out_zm["lev"].attrs["long_name"] = "pressure"

            return (out_1000, out_zm)

    except Exception as e:
        print(f"Error in {base_dir} year {year_int:04d}: {str(e)}")
        return None


# ================= 3. 主循环流水线 =================
if __name__ == "__main__":
    for exp in EXPERIMENTS:
        exp_name = exp["name"]
        base_dir = exp["base_dir"]
        apply_shift = exp["apply_shift"]
        file_template = exp["file_template"]

        data_out_dir = os.path.join(base_dir, "NAM")
        os.makedirs(data_out_dir, exist_ok=True)

        print("\n" + "🚀" * 25)
        print(f"正在处理实验组: {exp_name}")
        print("🚀" * 25)

        z3_files = glob.glob(os.path.join(base_dir, "Z3", "*.nc"))
        if not z3_files:
            z3_files = glob.glob(os.path.join(base_dir, "*.Z3.nc"))

        max_year = len(z3_files) if len(z3_files) > 0 else 250
        years_to_process = list(range(1, max_year + 1))

        print(f"--> 预计扫描 {len(years_to_process)} 年的数据...")
        args_list = [(y, base_dir, apply_shift, file_template) for y in years_to_process]

        with Pool(processes=CORES) as pool:
            results = pool.map(_process_wrapper, args_list)

        valid_results = [res for res in results if res is not None]
        if not valid_results:
            print(f"❌ 警告：未找到 {exp_name} 的有效数据，跳过该组。")
            continue

        z3_1000_all = xr.concat([res[0] for res in valid_results], dim="time").sortby("time")
        z3_zm_all = xr.concat([res[1] for res in valid_results], dim="time").sortby("time")
        z3_zm_all["lev"].attrs["units"] = "hPa"
        z3_zm_all["lev"].attrs["long_name"] = "pressure"
        
        del results, valid_results
        gc.collect()

        # =========================================================================
        # 第一节：计算 AO (保持不变)
        # =========================================================================
        print(f"🌟 开始处理 {exp_name} 的 AO 指数...")

        if not exp["use_reference"]:
            clim_1000 = z3_1000_all.groupby("time.dayofyear").mean("time")
            clim_1000_smooth = clim_1000.rolling(dayofyear=21, center=True).mean().bfill("dayofyear").ffill("dayofyear")
            daily_anom = (z3_1000_all.groupby("time.dayofyear") - clim_1000_smooth).drop_vars("dayofyear")

            monthly_anom = daily_anom.resample(time="1MS").mean().dropna(dim="time", how="all")
            coslat = np.clip(np.cos(np.deg2rad(monthly_anom.lat)), 0, None)
            wgts_2d = np.sqrt(coslat).broadcast_like(
                monthly_anom.isel(time=0).drop_vars("time", errors="ignore")
            )

            solver = Eof(monthly_anom, weights=wgts_2d.values)
            eof1 = solver.eofsAsCorrelation(neofs=1).squeeze()
            eof1_reg = solver.eofsAsCovariance(neofs=1, pcscaling=1).squeeze()
            pc1_raw = solver.pcs(npcs=1, pcscaling=0).squeeze()

            if eof1_reg.sel(lat=slice(75, 90)).mean() > 0:
                eof1 = eof1 * -1
                eof1_reg = eof1_reg * -1
                pc1_raw = pc1_raw * -1
                flip_factor = -1
            else:
                flip_factor = 1

            sigma_monthly = pc1_raw.std(dim="time")

            if exp["is_reference"]:
                REF_DATA["clim_1000_smooth"] = clim_1000_smooth
                REF_DATA["solver"] = solver
                REF_DATA["flip_factor"] = flip_factor
                REF_DATA["sigma_monthly"] = sigma_monthly
                print(f"--> [记录] {exp_name} 的基准 AO 气候态与求解器已缓存。")

            explained_var = solver.varianceFraction(neigs=1)[0].values * 100

            # ------- 绘制专属空间图 -------
            from cartopy.util import add_cyclic_point

            lon, lat = eof1.lon, eof1.lat
            eof1_cyclic, lon_cyclic = add_cyclic_point(eof1.values, coord=lon)

            fig1 = plt.figure(figsize=(8, 8))
            ax1 = plt.axes(projection=ccrs.NorthPolarStereo())
            ax1.set_extent([-180, 180, 20, 90], crs=ccrs.PlateCarree())
            ax1.coastlines(linewidth=1)
            ax1.add_feature(cfeature.BORDERS, linestyle=":", alpha=0.5)
            cf1 = ax1.contourf(
                lon_cyclic, lat, eof1_cyclic,
                levels=np.linspace(-1, 1, 21),
                transform=ccrs.PlateCarree(),
                cmap="RdBu_r",
                extend="both"
            )
            plt.colorbar(cf1, ax=ax1, orientation="horizontal", pad=0.05, aspect=40).set_label(
                "Correlation Coefficient", fontsize=12
            )
            plt.title(
                f"{exp_name} AO Spatial Pattern\nExplained Variance: {explained_var:.1f}%",
                fontsize=14, fontweight="bold", pad=15
            )
            plt.savefig(os.path.join(FIG_OUT_DIR, f"{exp_name}_AO_Pattern_Correlation.png"), dpi=300, bbox_inches="tight")
            plt.close(fig1)

            eof1_reg_cyclic, _ = add_cyclic_point(eof1_reg.values, coord=lon)
            fig2 = plt.figure(figsize=(9, 9))
            ax2 = plt.axes(projection=ccrs.NorthPolarStereo())
            ax2.set_extent([-180, 180, 20, 90], crs=ccrs.PlateCarree())
            ax2.coastlines(linewidth=1)
            ax2.add_feature(cfeature.BORDERS, linestyle=":", alpha=0.5)

            noaa_colors = [
                "#2d1e8f", "#4355c7", "#5a8ce6", "#7ebbf5", "#a3dcfb", "#c2f0fb",
                "#dff8fd", "#f0fcfd", "#ffffff", "#fefbd9", "#feea9e", "#fec762",
                "#f27932", "#b82522"
            ]
            bounds = [-45, -40, -35, -30, -25, -20, -15, -10, -5, 5, 10, 15, 20, 25, 30]
            cmap2 = mcolors.ListedColormap(noaa_colors)
            norm2 = mcolors.BoundaryNorm(bounds, cmap2.N)

            cf2 = ax2.contourf(
                lon_cyclic, lat, eof1_reg_cyclic,
                levels=bounds,
                transform=ccrs.PlateCarree(),
                cmap=cmap2,
                norm=norm2,
                extend="both"
            )
            cbar2 = plt.colorbar(
                cf2, ax=ax2, orientation="vertical", shrink=0.7, pad=0.05,
                ticks=[-40, -35, -30, -25, -20, -15, -10, -5, 5, 10, 15, 20, 25]
            )
            cbar2.ax.tick_params(labelsize=10)
            plt.title(
                f"{exp_name} Leading EOF ({explained_var:.0f}%) shown as\nregression map of 1000mb height (m)",
                fontsize=16, pad=20
            )
            plt.savefig(os.path.join(FIG_OUT_DIR, f"{exp_name}_AO_Regression_NOAA_Style.png"), dpi=300, bbox_inches="tight")
            plt.close(fig2)

            eof1_reg.name = "EOF1_Regression"
            eof1_reg.to_netcdf(os.path.join(data_out_dir, f"{exp_name}_EOF1_Regression.nc"))

            eof1.name = "EOF1_Correlation"
            eof1.to_netcdf(os.path.join(data_out_dir, f"{exp_name}_EOF1_Correlation.nc"))

        else:
            print(f"--> [核心] 正在应用基准组 (Couple) 的气候态与 EOF 求解器...")
            ref_clim = REF_DATA["clim_1000_smooth"]
            ref_solver = REF_DATA["solver"]
            flip_factor = REF_DATA["flip_factor"]
            sigma_monthly = REF_DATA["sigma_monthly"]

            daily_anom = (z3_1000_all.groupby("time.dayofyear") - ref_clim).drop_vars("dayofyear")
            solver = ref_solver
            print("--> (BWCN 组借用参考组模态，跳过绘制专属 EOF 空间图)")

        pseudo_pcs_raw = solver.projectField(daily_anom, neofs=1, eofscaling=0).squeeze() * flip_factor
        ao_index_daily = pseudo_pcs_raw / sigma_monthly

        csv_path = os.path.join(data_out_dir, f"{exp_name}_Daily_AO_Index.csv")
        pd.DataFrame({
            "Date": ao_index_daily.time.values,
            "AO_Index": ao_index_daily.values
        }).to_csv(csv_path, index=False)
        print(f"✅ {exp_name} AO 数据已成功归档至: {data_out_dir}")

        # =========================================================================
        # 第二节：计算 NAM (全新 Monthly EOF + DOY 投影法)
        # =========================================================================
        print(f"🌟 开始处理 {exp_name} 的多层垂直 NAM 指数 (EOF 投影法)...")

        nam_daily_levels = []

        if not exp["use_reference"]:
            # 1. 计算用于训练 EOF 的月均异常
            z3_zm_monthly = z3_zm_all.resample(time="1MS").mean().dropna(dim="time", how="all")
            clim_mon_zm = z3_zm_monthly.groupby("time.month").mean("time")
            anom_mon_zm = z3_zm_monthly.groupby("time.month") - clim_mon_zm

            # 2. 计算用于投影的 DOY 平滑异常
            clim_doy_zm = z3_zm_all.groupby("time.dayofyear").mean("time")
            clim_doy_zm_smooth = clim_doy_zm.rolling(dayofyear=21, center=True).mean().bfill("dayofyear").ffill("dayofyear")
            anom_doy_zm = (z3_zm_all.groupby("time.dayofyear") - clim_doy_zm_smooth).drop_vars("dayofyear")

            # Xarray 专用权重 (与纬度挂钩)
            coslat_da = np.clip(np.cos(np.deg2rad(z3_zm_all.lat)), 0, None)
            wgts_1d = np.sqrt(coslat_da)

            nam_ref_data_levels = {}

            print("--> 正在逐层进行 EOF 分解与每日投影...")
            for lev in z3_zm_all.lev.values:
                # 提取单层数据
                da_mon = anom_mon_zm.sel(lev=lev)
                da_day = anom_doy_zm.sel(lev=lev)

                # 训练 EOF
                solver_nam = Eof(da_mon, weights=wgts_1d.values)
                eof1_nam = solver_nam.eofs(neofs=1, eofscaling=0).squeeze()
                pc1_raw_nam = solver_nam.pcs(npcs=1, pcscaling=0).squeeze()

                # 检查极性 (80N附近为负)
                if float(eof1_nam.sel(lat=80.0, method="nearest").values) > 0:
                    flip_nam = -1
                else:
                    flip_nam = 1

                pc1_raw_nam = pc1_raw_nam * flip_nam
                pc_mean = pc1_raw_nam.mean(dim="time")
                pc_std = pc1_raw_nam.std(dim="time")

                # 记录本层的基准数据
                nam_ref_data_levels[float(lev)] = {
                    "solver": solver_nam,
                    "flip_factor": flip_nam,
                    "pc_mean": pc_mean,
                    "pc_std": pc_std
                }

                # 每日数据投影
                pseudo_pcs_nam = solver_nam.projectField(da_day, neofs=1, eofscaling=0).squeeze() * flip_nam
                nam_day = (pseudo_pcs_nam - pc_mean) / pc_std
                nam_daily_levels.append(nam_day)

            if exp["is_reference"]:
                REF_DATA["clim_doy_zm_smooth_nam"] = clim_doy_zm_smooth
                REF_DATA["nam_levels"] = nam_ref_data_levels
                print(f"--> [记录] {exp_name} 的基准 NAM DOY气候态与逐层 EOF 求解器已缓存。")

        else:
            print(f"--> [核心] 正在应用基准组的 NAM 气候态与逐层 EOF 模态...")
            ref_clim_doy_zm = REF_DATA["clim_doy_zm_smooth_nam"]
            ref_nam_levels = REF_DATA["nam_levels"]

            # 使用基准组的气候态计算 DOY 异常
            anom_doy_zm = (z3_zm_all.groupby("time.dayofyear") - ref_clim_doy_zm).drop_vars("dayofyear")

            for lev in z3_zm_all.lev.values:
                da_day = anom_doy_zm.sel(lev=lev)
                
                # 读取该层的基准模态特征
                ref_dict = ref_nam_levels[float(lev)]
                solver_nam = ref_dict["solver"]
                flip_nam = ref_dict["flip_factor"]
                pc_mean = ref_dict["pc_mean"]
                pc_std = ref_dict["pc_std"]

                # 每日数据投影
                pseudo_pcs_nam = solver_nam.projectField(da_day, neofs=1, eofscaling=0).squeeze() * flip_nam
                nam_day = (pseudo_pcs_nam - pc_mean) / pc_std
                nam_daily_levels.append(nam_day)

        # 把列表里的各层 DataArray 沿着 lev 维度拼接起来
        nam_vertical = xr.concat(nam_daily_levels, dim=z3_zm_all.lev)
        nam_vertical.name = "NAM_Vertical"
        nam_vertical.attrs["long_name"] = f"EOF-based Vertical NAM index ({exp_name})"
        nam_vertical["lev"].attrs["units"] = "hPa"
        nam_vertical["lev"].attrs["long_name"] = "pressure"

        nc_nam_path = os.path.join(data_out_dir, f"{exp_name}_Vertical_NAM.nc")
        nam_vertical.to_netcdf(nc_nam_path)
        print(f"✅ {exp_name} 全量多层垂直 NAM 已保存至: {nc_nam_path}")

        del z3_1000_all, z3_zm_all, daily_anom, nam_vertical
        gc.collect()

    print("\n🎉 全部 3 组跨实验对比计算与归档任务圆满完成！")

In [ ]:
import os
import xarray as xr
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# ================= 1. 核心配置区 =================
# 实验组名称
EXP_NAME = "BWCN"

# 数据输入目录 (指向你跑完新 NAM 的输出文件夹)
DATA_DIR = f"/mnt/soclim0/public_data/weiji/{EXP_NAME}/NAM"

# 图片输出目录
FIG_OUT_DIR = "/home/weiji/restart_exam/code/20260315NAM/resul/"
os.makedirs(FIG_OUT_DIR, exist_ok=True)

# 输入文件完整路径
NAM_FILE = os.path.join(DATA_DIR, f"{EXP_NAME}_Vertical_NAM.nc")
AO_FILE = os.path.join(DATA_DIR, f"{EXP_NAME}_Daily_AO_Index.csv")

# 绘图的时间窗口 (模拟参考图中横跨整个冬季的视图)
START_DATE = "0007-11-01"
END_DATE = "0008-05-31"

print(f"📂 正在读取 {EXP_NAME} 组的 NAM 和 AO 数据...")

# ================= 2. 数据读取与精准截取 =================
# 读取多层垂直 NAM 数据
ds_nam = xr.open_dataset(NAM_FILE, use_cftime=True) # 模式数据加上 use_cftime 保平安
nam_da = ds_nam["NAM_Vertical"] # 新流水线的变量名

# 利用 xarray 的 slice 完美截取时间段 (支持 cftime 字符串切片)
nam_sub = nam_da.sel(time=slice(START_DATE, END_DATE))

# 读取地面 AO CSV 数据
ao_df = pd.read_csv(AO_FILE)

# 确保 CSV 中的日期格式干净，并按字符串切片出相同的日期范围 (极其重要，避开年份太小的限制)
ao_df['Date_str'] = ao_df['Date'].astype(str).str[:10]
mask = (ao_df['Date_str'] >= START_DATE) & (ao_df['Date_str'] <= END_DATE)
ao_sub = ao_df[mask].copy()

# 提取用于绘图的数组
time_arr = nam_sub.time.values
ao_vals = ao_sub['AO_Index'].values

print("🎨 正在渲染平流层-对流层耦合 (Time-Height) 剖面图...")

# ================= 3. 高级布局与绘图 (解决严丝合缝对齐问题) =================
fig = plt.figure(figsize=(10, 6))

gs = fig.add_gridspec(nrows=2, ncols=2, 
                      height_ratios=[2.0, 1.0], 
                      width_ratios=[1, 0.02], 
                      hspace=0.08, wspace=0.02)

# -------------------------------------------------------------
# 面板 (a): NAM Time-Height 等值线图 (放在左上角 [0, 0])
# -------------------------------------------------------------
ax1 = fig.add_subplot(gs[0, 0])
cax = fig.add_subplot(gs[0, 1]) 

levels_c = np.arange(-4.5, 5.0, 0.5)

cf = nam_sub.plot.contourf(
    ax=ax1,
    x="time",
    y="lev",        # 模式数据的垂直坐标通常叫 lev
    levels=levels_c,
    cmap="RdBu",   
    extend="both",
    add_colorbar=False
)

ax1.set_yscale("log")
ax1.invert_yaxis()
ax1.set_ylim(1000, 1) 
ax1.set_yticks([1000, 300, 100, 30, 10, 3, 1])
ax1.get_yaxis().set_major_formatter(plt.ScalarFormatter())

ax1.set_ylabel("Pressure [hPa]", fontsize=16)
ax1.set_xlabel("")
ax1.tick_params(labelbottom=False)  
ax1.set_title("") 
ax1.text(0.01, 0.92, "(a)", transform=ax1.transAxes, fontsize=16, fontweight="bold",
         bbox=dict(facecolor='white', alpha=0.8, edgecolor='none', pad=2))

cbar = fig.colorbar(cf, cax=cax)
cbar.set_label("NAM Index", fontsize=14)

# -------------------------------------------------------------
# 面板 (b): 地面 AO 时间序列折线图 (放在左下角 [1, 0])
# -------------------------------------------------------------
ax2 = fig.add_subplot(gs[1, 0], sharex=ax1)

ax2.plot(time_arr, ao_vals, color='black', lw=1.8)
ax2.axhline(0, color='gray', lw=1.2, linestyle='-')

ax2.set_ylabel(r"$AO_{\mathrm{WACCM}}$", fontsize=16)
ax2.set_ylim(-4.5, 4.5)
ax2.set_yticks([-3, 0, 3])
ax2.text(0.01, 0.82, "(b)", transform=ax2.transAxes, fontsize=16, fontweight="bold",
         bbox=dict(facecolor='white', alpha=0.8, edgecolor='none', pad=2))
ax2.grid(True, linestyle=':', alpha=0.6)

# ================= 4. 标题与保存 =================
fig.suptitle(f'Time series of NAM and AO indices ({EXP_NAME}: Yr 0007-0008)', 
             fontsize=18, fontweight='bold', y=0.95)

out_fig = os.path.join(FIG_OUT_DIR, f"{EXP_NAME}_NAM_AO_TimeHeight_Yr0007_0008.png")
plt.savefig(out_fig, dpi=300, bbox_inches='tight')
print(f"✅ 绝美且绝对对齐的剖面图已保存至: {out_fig}")

plt.show()

ds_nam.close()